# Проверочная работа «Классификация текстов»

**Тема:** наивный байесовский классификатор, логистическая регрессия, линейный метод опорных векторов (SVM).
**Данные:** 400 твитов Дональда Трампа и Джастина Трюдо, собранных в ноябре 2017 года, файл `tweets.csv`.
**Задача:** по тексту твита определить автора.

**Фамилия, имя, группа:** _впишите здесь_

## Правила

- Время выполнения 80 минут. Всего 40 баллов. «Отлично» от 34 баллов, «хорошо» от 27, «удовлетворительно» от 20.
- В ячейках с кодом замените многоточия `...` своим решением. Имена переменных не меняйте: на них опираются проверочные ячейки.
- После каждого задания стоит проверочная ячейка. Она должна выполниться без ошибок. Проверка контролирует форму результата, правильность чисел оценивает преподаватель.
- На вопросы отвечайте письменно в ячейке «Ответ», двумя-четырьмя фразами, со ссылкой на полученные числа.
- Разрешены конспект лекции 2 и документация scikit-learn. Перед сдачей выполните блокнот целиком: «Kernel → Restart & Run All».

## Подготовка

Ячейки этого раздела готовы, их нужно только выполнить. В них импортируются библиотеки и определяются две функции для рисунков: `plot_confusion` рисует матрицу ошибок, `plot_top_words` показывает слова с наибольшими по модулю весами линейной модели.

In [ ]:
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

SEED = 53
random.seed(SEED); np.random.seed(SEED)
CLASSES = ["Donald J. Trump", "Justin Trudeau"]
BLUE, ORANGE = "#2a78d6", "#eb6834"

In [ ]:
def plot_confusion(y_true, y_pred, title, ax=None):
    # Матрица ошибок: строки показывают настоящего автора, столбцы предсказанного.
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    ax = ax or plt.subplots(figsize=(3.6, 3.2))[1]
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.sum() / 2)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14, color="white" if cm[i, j] > cm.sum() / 4 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Трамп", "Трюдо"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["Трамп", "Трюдо"])
    ax.set_xlabel("предсказано"); ax.set_ylabel("на самом деле"); ax.set_title(title)
    return cm


def plot_top_words(classifier, vectorizer, title, n=15, ax=None):
    # Слова с наибольшими по модулю весами линейной модели. Отрицательный вес сдвигает решение
    # к первому классу (Трамп), положительный ко второму (Трюдо).
    words = np.array(vectorizer.get_feature_names_out())
    weights = classifier.coef_[0]
    order = np.r_[np.argsort(weights)[:n], np.argsort(weights)[-n:]]
    ax = ax or plt.subplots(figsize=(6, 6))[1]
    ax.barh(words[order], weights[order], color=[ORANGE if w > 0 else BLUE for w in weights[order]])
    ax.axvline(0, color="gray", lw=1); ax.set_title(title); ax.set_xlabel("вес слова: влево Трамп, вправо Трюдо")
    return list(zip(words[order], np.round(weights[order], 3)))

## Задание 1. Данные и разбиение (3 балла)

1. Загрузите файл `tweets.csv` в таблицу `tweet_df`. В таблице три столбца: `id`, `author`, `status` (текст твита).
2. Постройте столбчатую диаграмму числа твитов каждого автора.
3. Разделите данные на обучающую и тестовую части функцией `train_test_split`: тексты берутся из столбца `status`, метки из столбца `author`, доля тестовой части 0.33, `random_state=SEED`, разбиение стратифицированное (`stratify=y`).

In [ ]:
# Загрузите данные
tweet_df = ...

# Диаграмма числа твитов по авторам
...

# Целевая переменная и разбиение
y = ...
X_train, X_test, y_train, y_test = ...

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert list(tweet_df.columns) == ["id", "author", "status"] and len(tweet_df) == 400
assert len(X_train) == 268 and len(X_test) == 132, "проверьте долю тестовой части"
assert abs((y_train == CLASSES[0]).mean() - 0.5) < 0.01, "разбиение должно быть стратифицированным"
print("Задание 1: форма результата верна")

**Вопрос 1.1.** Зачем разбиение делается стратифицированным? Что могло бы произойти без параметра `stratify` на выборке из 400 твитов?

**Ответ:**

_напишите ответ здесь_

## Задание 2. Векторизация (4 балла)

Создайте два векторизатора с одинаковыми параметрами: английские стоп-слова (`stop_words="english"`), слово должно встретиться не менее чем в 3 твитах (`min_df=3`) и не более чем в 90 % твитов (`max_df=0.9`).

1. `count_vectorizer` класса `CountVectorizer` даёт матрицы счётчиков `count_train` и `count_test`.
2. `tfidf_vectorizer` класса `TfidfVectorizer` даёт матрицы весов TF-IDF `tfidf_train` и `tfidf_test`.

Словарь строится только по обучающей части: метод `fit_transform` применяется к `X_train`, метод `transform` к `X_test`.

In [ ]:
count_vectorizer = ...
count_train = ...
count_test = ...

tfidf_vectorizer = ...
tfidf_train = ...
tfidf_test = ...

print("Размер словаря:", ...)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert count_train.shape[0] == 268 and count_test.shape[0] == 132
assert count_train.shape[1] == count_test.shape[1] == tfidf_train.shape[1] == tfidf_test.shape[1], "у матриц должен быть общий словарь"
assert 100 < count_train.shape[1] < 600, "проверьте параметры min_df и max_df"
assert abs(tfidf_train[0].power(2).sum() - 1) < 1e-6 and count_train.max() >= 1
print("Задание 2: форма результата верна, размер словаря", count_train.shape[1])

**Вопрос 2.1.** Почему словарь нельзя строить по всем 400 твитам сразу?

**Вопрос 2.2.** Чем значение в матрице `tfidf_train` отличается от значения в матрице `count_train` для одного и того же слова и твита?

**Ответ:**

_напишите ответ здесь_

## Задание 3. Точка отсчёта (3 балла)

Прежде чем обучать модели, нужно знать, какое качество даёт решение без модели. Обучите `DummyClassifier(strategy="most_frequent")` и посчитайте на тестовой части долю верных ответов (`accuracy_score`) и macro-F1 (`f1_score` с параметром `average="macro"`).

Результаты всех моделей складывайте в словарь `results`: ключом служит название модели, значением пара (доля верных ответов, macro-F1).

In [ ]:
results = {}

dummy = ...
dummy_pred = ...
results["Постоянный ответ"] = (..., ...)
print(results)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
acc, f1 = results["Постоянный ответ"]
assert 0.45 <= acc <= 0.55 and 0.3 <= f1 <= 0.4, "проверьте стратегию и параметр average"
print("Задание 3: форма результата верна")

**Вопрос 3.1.** Почему у постоянного ответа доля верных ответов около 0.5, а macro-F1 около 0.33? Какая из двух мер честнее описывает такую модель?

**Ответ:**

_напишите ответ здесь_

## Задание 4. Наивный байесовский классификатор (5 баллов)

Обучите две модели `MultinomialNB` с параметрами по умолчанию: `count_nb` на счётчиках и `tfidf_nb` на весах TF-IDF. Для каждой получите предсказания на тестовой части и добавьте результат в словарь `results` под ключами `"NB, счётчики"` и `"NB, TF-IDF"`. Постройте матрицы ошибок обеих моделей функцией `plot_confusion`.

In [ ]:
count_nb = ...
count_nb_pred = ...
results["NB, счётчики"] = (..., ...)

tfidf_nb = ...
tfidf_nb_pred = ...
results["NB, TF-IDF"] = (..., ...)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
plot_confusion(y_test, ..., "NB, счётчики", ax=axes[0])
plot_confusion(y_test, ..., "NB, TF-IDF", ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
for k in ("NB, счётчики", "NB, TF-IDF"):
    assert 0.75 <= results[k][0] <= 1.0, f"{k}: качество слишком низкое, проверьте обучение"
assert len(count_nb_pred) == len(tfidf_nb_pred) == 132
print("Задание 4: форма результата верна")

**Вопрос 4.1.** По матрице ошибок модели на TF-IDF: сколько твитов Трюдо приписано Трампу и сколько твитов Трампа приписано Трюдо?

**Вопрос 4.2.** В чём состоит «наивное» предположение метода? Приведите пример двух слов из твитов, для которых оно нарушается.

**Вопрос 4.3.** Какую роль играет параметр `alpha` модели `MultinomialNB`? Что произошло бы при `alpha=0` с твитом, в котором есть слово, не встречавшееся у одного из авторов?

**Ответ:**

_напишите ответ здесь_

## Задание 5. Логистическая регрессия (4 балла)

Обучите `LogisticRegression(C=10, max_iter=1000)` на весах TF-IDF, сохраните модель в `tfidf_lr`, предсказания в `tfidf_lr_pred`, результат в `results["LogReg, TF-IDF"]`.

Затем выведите вероятности классов (`predict_proba`) для первых пяти тестовых твитов вместе с текстом и настоящим автором.

In [ ]:
tfidf_lr = ...
tfidf_lr_pred = ...
results["LogReg, TF-IDF"] = (..., ...)

# Вероятности классов для первых пяти тестовых твитов
proba = ...
...

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert 0.8 <= results["LogReg, TF-IDF"][0] <= 1.0
assert proba.shape == (5, 2) and np.allclose(proba.sum(axis=1), 1)
print("Задание 5: форма результата верна")

**Вопрос 5.1.** Как логистическая регрессия получает вероятность класса из весов слов? Запишите формулу и поясните обозначения.

**Вопрос 5.2.** Среди пяти твитов найдите тот, в котором модель наименее уверена. Чем это можно объяснить, глядя на текст?

**Ответ:**

_напишите ответ здесь_

## Задание 6. Линейный SVM (4 балла)

Обучите `LinearSVC(C=1)` на весах TF-IDF, сохраните модель в `tfidf_svc`, предсказания в `tfidf_svc_pred`, результат в `results["LinearSVC, TF-IDF"]`.

Выведите значения решающей функции (`decision_function`) для тех же пяти тестовых твитов.

In [ ]:
tfidf_svc = ...
tfidf_svc_pred = ...
results["LinearSVC, TF-IDF"] = (..., ...)

svc_scores = ...
print(svc_scores)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert 0.8 <= results["LinearSVC, TF-IDF"][0] <= 1.0
assert svc_scores.shape == (5,)
print("Задание 6: форма результата верна")

**Вопрос 6.1.** Как по значению решающей функции определить предсказанный класс? Что означает значение, близкое к нулю?

**Вопрос 6.2.** Что такое опорные векторы и почему удаление твита, лежащего далеко от границы, не изменит модель SVM?

**Ответ:**

_напишите ответ здесь_

## Задание 7. Сравнение моделей (5 баллов)

1. Преобразуйте словарь `results` в таблицу `results_df` со столбцами `accuracy` и `macro_f1` и постройте столбчатую диаграмму macro-F1 всех моделей.
2. Постройте рядом матрицы ошибок трёх моделей на TF-IDF: наивного Байеса, логистической регрессии, SVM.
3. Оценка по одному разбиению зависит от случая. Для трёх моделей на TF-IDF выполните кросс-валидацию на всех 400 твитах: 5 частей, `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`, мера `f1_macro`. Векторизатор должен обучаться внутри каждой части, поэтому модель оборачивается в конвейер `make_pipeline(TfidfVectorizer(...), модель)`. Сохраните в словарь `cv_scores` массив из пяти значений для каждой модели и выведите среднее и стандартное отклонение.

In [ ]:
results_df = ...
...

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
...
plt.tight_layout(); plt.show()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = {}
for name, model in [("NB", MultinomialNB()), ("LogReg", LogisticRegression(C=10, max_iter=1000)), ("LinearSVC", LinearSVC(C=1))]:
    pipe = ...
    cv_scores[name] = ...
    print(name, ...)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert set(results_df.columns) == {"accuracy", "macro_f1"} and len(results_df) == 5
assert all(len(v) == 5 for v in cv_scores.values()) and set(cv_scores) == {"NB", "LogReg", "LinearSVC"}
assert all(0.75 <= v.mean() <= 1 for v in cv_scores.values())
print("Задание 7: форма результата верна")

**Вопрос 7.1.** Какая модель лучше по тестовой части и какая по кросс-валидации? Можно ли по вашим числам утверждать, что одна из трёх моделей лучше остальных? Обоснуйте через разброс между частями.

**Вопрос 7.2.** Почему при кросс-валидации векторизатор помещён внутрь конвейера?

**Ответ:**

_напишите ответ здесь_

## Задание 8. Что выучила модель (5 баллов)

1. Функцией `plot_top_words` покажите слова с наибольшими по модулю весами для логистической регрессии и для SVM (два рисунка рядом).
2. Среди слов Трюдо есть французские служебные слова, пометка пересылки `rt` и имена аккаунтов. Модель различает авторов не только по содержанию. Проверьте, насколько результат зависит от таких признаков: напишите функцию `clean_tweet`, которая удаляет из текста ссылки (всё, что начинается с `http`) и упоминания (`@имя`), и повторите кросс-валидацию для `LinearSVC` на очищенных текстах с добавлением французских стоп-слов из списка `FRENCH_STOP`. Сохраните пять значений в `cv_clean`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
...
plt.tight_layout(); plt.show()

FRENCH_STOP = ["le", "la", "les", "de", "des", "du", "et", "en", "un", "une", "au", "aux", "pour", "nous", "notre", "nos",
               "que", "qui", "dans", "sur", "est", "sont", "ce", "ces", "avec", "plus", "il", "elle", "je", "vous", "ont", "été"]


def clean_tweet(text):
    ...


stop_all = list(TfidfVectorizer(stop_words="english").get_stop_words()) + FRENCH_STOP
pipe_clean = make_pipeline(TfidfVectorizer(preprocessor=..., stop_words=stop_all, min_df=3, max_df=0.9), LinearSVC(C=1))
cv_clean = ...
print("LinearSVC без очистки:", ...)
print("LinearSVC с очисткой: ", ...)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert "http" not in clean_tweet("Great day https://t.co/abc @WhiteHouse") and "@" not in clean_tweet("hello @user")
assert len(cv_clean) == 5 and 0.6 <= cv_clean.mean() <= 1
print("Задание 8: форма результата верна")

**Вопрос 8.1.** Назовите по три слова с наибольшим весом для каждого автора. Какие из них отражают содержание твитов, а какие особенности оформления или языка?

**Вопрос 8.2.** Как изменилось качество после очистки и что это говорит о модели без очистки? Какую из двух моделей вы бы использовали, если задача состоит в изучении тем, о которых пишут политики?

**Ответ:**

_напишите ответ здесь_

## Задание 9. Свои твиты (3 балла)

Напишите на английском два твита по 10–20 слов: один в стиле Трампа, другой в стиле Трюдо. Используйте слова с большими весами из задания 8. Получите предсказания трёх моделей на TF-IDF и вероятность по логистической регрессии.

Затем напишите третий твит, на котором модели расходятся или логистическая регрессия даёт вероятность между 0.4 и 0.6.

In [ ]:
my_tweets = ["...", "...", "..."]

my_vec = ...
for name, model in [("NB", tfidf_nb), ("LogReg", tfidf_lr), ("LinearSVC", tfidf_svc)]:
    print(name, ...)
print("P(Трюдо) по логистической регрессии:", ...)

In [ ]:
# Проверка. Ячейку не менять: она должна выполняться без ошибок.
assert len(my_tweets) == 3 and all(len(t.split()) >= 5 for t in my_tweets), "нужно три твита"
assert my_vec.shape == (3, tfidf_train.shape[1])
print("Задание 9: форма результата верна")

**Вопрос 9.1.** Почему третий твит оказался трудным для моделей? Какие слова в нём сдвигают решение в разные стороны?

**Ответ:**

_напишите ответ здесь_

## Задание 10. Вопросы по методам (4 балла)

По одному баллу за вопрос. Ответ в две-четыре фразы.

**10.1.** Наивный Байес строит модель каждого класса, логистическая регрессия и SVM сразу строят границу. Какое практическое следствие это имеет при очень малом числе обучающих примеров?

**10.2.** Параметр `C` есть и у логистической регрессии, и у SVM. Что произойдёт с весами слов и с качеством на обучающей части при очень малом `C` (например, 0.001)?

**10.3.** Объект классифицирован верно и лежит далеко от границы. Чему для него равна шарнирная функция потерь SVM и чему близка логарифмическая функция потерь логистической регрессии? Что из этого следует?

**10.4.** В данных 95 % твитов одного автора. Какая из мер, доля верных ответов или macro-F1, подходит для оценки модели и почему?

**Ответ:**

_напишите ответ здесь_

## Перед сдачей

- [ ] Блокнот выполняется целиком без ошибок после перезапуска ядра.
- [ ] Все проверочные ячейки напечатали «форма результата верна».
- [ ] На каждый вопрос дан письменный ответ со ссылкой на полученные числа.
- [ ] В начале блокнота указаны фамилия, имя и группа.